# HealSense — PhysioNet 2019 Training Notebook

**Self-contained.** No repo clone, no uploads, no setup. Run the cells top to bottom.

**Runtime → Change runtime type → GPU** (T4 on Colab, P100 on Kaggle).

| Setting | Default | Why |
|---|---|---|
| `SEQ_LEN` | **24** | Only 4% of patients have ≥60 hourly rows — 60 would discard 96% of the cohort |
| `HORIZON` | **6** | Predicts deterioration in the next 6 hours (standard early-warning convention) |
| `PRESET` | **medium** | ~150k params. See the parameter note below before choosing `large` |

**Expected run time:** ~25–40 min for the full dataset on a T4/P100.
**Outputs:** `lstm_best.h5`, `scaler.pkl`, `label_encoder.pkl`, `imputer.pkl`, `model_meta.json` — all drop-in compatible with the HealSense backend.

## 0 · Configuration
Edit only this cell.

In [ ]:
import os, sys, json, glob, time, subprocess
from pathlib import Path

# ─────────────── EDIT ME ───────────────
SEQ_LEN   = 24        # history window in hours
HORIZON   = 6         # K: hours ahead to predict
PRESET    = "medium"  # "small" | "medium" | "large"  (see table below)
EPOCHS    = 25
BATCH     = 256
SEED      = 42

N_PATIENTS_LIMIT = None   # set e.g. 5000 for a fast first pass; None = use everything
USE_DRIVE        = True   # checkpoint to Google Drive so a disconnect never loses progress
# ───────────────────────────────────────

MODEL_VERSION = "v1.0.2"
PRESETS = {
    #  LSTM units    dense  dropout              approx params (input 24x10)
    "small":  {"lstm": [64, 32],   "dense": 32,  "dr": [0.30, 0.20, 0.10]},  # ~31k
    "medium": {"lstm": [128, 64],  "dense": 64,  "dr": [0.35, 0.25, 0.20]},  # ~150k
    "large":  {"lstm": [256, 128], "dense": 128, "dr": [0.40, 0.30, 0.30]},  # ~600k
}
CFG = PRESETS[PRESET]

IS_KAGGLE = os.path.exists("/kaggle")
# Kaggle ships a partial google.colab shim, so "google.colab in sys.modules"
# is TRUE on Kaggle too. Check Kaggle FIRST or the data path resolves to
# /content/... on a Kaggle box.
IS_COLAB = (not IS_KAGGLE) and ("google.colab" in sys.modules or os.path.exists("/content"))

print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'local'}")
print(f"Preset      : {PRESET}  {CFG}")
print(f"Seq len     : {SEQ_LEN}   Horizon: {HORIZON}h")

try:
    import tensorflow as tf
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"], check=True)
    import tensorflow as tf

# CPU thread tuning — matters a lot when there is no GPU
N_CPU = os.cpu_count() or 2
try:
    tf.config.threading.set_intra_op_parallelism_threads(N_CPU)
    tf.config.threading.set_inter_op_parallelism_threads(max(1, N_CPU // 2))
except Exception:
    pass

gpus = tf.config.list_physical_devices("GPU")
print(f"TensorFlow  : {tf.__version__}")
print(f"CPUs        : {N_CPU}")
print(f"GPU         : {[g.name for g in gpus] if gpus else 'NONE (CPU training)'}")
if not gpus:
    print("ℹ️  No GPU. That is FINE for this model — CPU runs in roughly 25-50 min")
    print("   for the full dataset. See the timing estimate printed before training.")
    print("   (Kaggle/Colab unlock GPUs only after phone verification.)")

## 1 · Get the data

Handles all three layouts automatically:
- **Kaggle**: dataset already mounted at `/kaggle/input/...` → zero download
- **Colab**: parallel download from the PhysioNet S3 mirror (~4 min, 14× faster than the website)
- **Drive**: reuses a previously downloaded copy

Add the dataset on Kaggle: search **`salikhussaini49/prediction-of-sepsis`** → *Add Input*.

In [ ]:
DATA_ROOT = Path("/content/healsense_data" if IS_COLAB else "/kaggle/working/healsense_data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

def find_existing_psv():
    """Look for .psv/.csv patient files anywhere already available."""
    search_roots = []
    if IS_KAGGLE:
        search_roots.append("/kaggle/input")
    if IS_COLAB and USE_DRIVE:
        try:
            from google.colab import drive
            if not os.path.ismount("/content/drive"):
                drive.mount("/content/drive")
            search_roots.append("/content/drive/MyDrive/healsense/data")
        except Exception as e:
            print(f"Drive mount skipped: {e}")
    search_roots.append(str(DATA_ROOT))

    for root in search_roots:
        if not os.path.isdir(root):
            continue
        hits = glob.glob(os.path.join(root, "**", "p*.psv"), recursive=True)
        if not hits:
            hits = glob.glob(os.path.join(root, "**", "p*.csv"), recursive=True)
        if hits:
            print(f"Found {len(hits)} patient files under {root}")
            return sorted(hits)
    return []

files = find_existing_psv()

if not files:
    print("No local copy found — downloading from the PhysioNet S3 mirror...")
    import concurrent.futures as cf, urllib.request

    BASE = "https://physionet-open.s3.amazonaws.com/challenge-2019/1.0.0/training"
    N_A, N_B = 20336, 20000
    jobs = [("training_setA", f"p{i:06d}.psv", N_A) for i in range(1, N_A + 1)] + \
           [("training_setB", f"p{i:06d}.psv", N_B) for i in range(1, N_B + 1)]
    if N_PATIENTS_LIMIT:
        jobs = jobs[:N_PATIENTS_LIMIT]

    def fetch(job):
        subset, name, _ = job
        dest = DATA_ROOT / subset / name
        if dest.exists() and dest.stat().st_size > 200:
            return str(dest)
        dest.parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(f"{BASE}/{subset}/{name}", dest)
            if dest.stat().st_size < 200:   # 404 body
                dest.unlink(); return None
            return str(dest)
        except Exception:
            if dest.exists():
                dest.unlink()
            return None

    t0 = time.time()
    done = 0
    with cf.ThreadPoolExecutor(max_workers=48) as ex:
        for res in ex.map(fetch, jobs, chunksize=64):
            done += 1
            if done % 5000 == 0:
                print(f"  {done}/{len(jobs)} files | {time.time()-t0:.0f}s")
    files = sorted(str(p) for p in DATA_ROOT.rglob("*.psv") if p.stat().st_size > 200)

print(f"\nTotal patient files: {len(files)}")
if not files:
    raise RuntimeError("No data. On Kaggle add the dataset as input; on Colab re-run this cell.")

## 2 · Preprocess into windowed sequences

Labels are **future-window**: the worst severity in the `HORIZON` hours *after* the window ends. This is what removes the circularity in the original v1 pipeline.

In [ ]:
import numpy as np, pandas as pd

FEATURES = ["HR", "O2Sat", "Temp", "SBP", "DBP"]
CLASS_NAMES = ["normal", "warning", "critical"]
NORMAL, WARNING, CRITICAL = 0, 1, 2

# Mirror of backend/api/services/fusion_engine.py::evaluate_deterministic_rules
def severity_of_row(row):
    def val(c):
        v = row.get(c)
        try:
            v = float(v)
        except (TypeError, ValueError):
            return None
        return None if np.isnan(v) else v

    spo2, hr, temp, sbp, dbp = val("O2Sat"), val("HR"), val("Temp"), val("SBP"), val("DBP")
    if all(v is None for v in (spo2, hr, temp, sbp, dbp)):
        return None
    if (spo2 is not None and spo2 < 92) or \
       (hr is not None and (hr > 150 or hr < 40)) or \
       (sbp is not None and sbp >= 180) or (dbp is not None and dbp >= 120):
        return CRITICAL
    if (spo2 is not None and spo2 < 95) or \
       (hr is not None and (hr > 120 or hr < 50)) or \
       (temp is not None and (temp >= 38.0 or temp < 35.0)):
        return WARNING
    return NORMAL


def patient_windows(df, seq_len, horizon, min_valid=0.20):
    if len(df) < seq_len + 1:
        return None, None, None
    vals = df[FEATURES].to_numpy(dtype=np.float32)
    sev = np.full(len(df), -1, dtype=np.int8)
    for i in range(len(df)):
        s = severity_of_row(df.iloc[i])
        if s is not None:
            sev[i] = s

    X, y, idx = [], [], []
    for end in range(seq_len - 1, len(df)):
        fut = sev[end + 1: end + 1 + horizon]
        fut = fut[fut >= 0]
        if fut.size == 0:
            continue
        win = vals[end - seq_len + 1: end + 1]
        if np.mean(~np.isnan(win)) < min_valid:
            continue
        X.append(win); y.append(int(fut.max())); idx.append(end)
    if not X:
        return None, None, None
    return np.stack(X).astype(np.float32), np.asarray(y, np.int8), np.asarray(idx)


if N_PATIENTS_LIMIT:
    files = files[:N_PATIENTS_LIMIT]

# ── Robust loader ────────────────────────────────────────────────────────────
# Handles BOTH layouts that appear on Kaggle/Colab mirrors:
#   (a) one .psv file per patient          -> patient id = filename
#   (b) one combined CSV with a patient id column -> split by that column
# Without (b) a combined 1.55M-row CSV would be read as a SINGLE patient and
# sequences would bleed across patient boundaries.
PATIENT_ID_COLS = ["patient_id", "Patient_ID", "PatientID", "patient", "pid", "Patient"]


def load_patient_frames(paths):
    # Yield (patient_id, DataFrame) preserving original row order.
    for fp in paths:
        ext = fp.lower().rsplit(".", 1)[-1]
        sep = "|" if ext == "psv" else ","
        try:
            df = pd.read_csv(fp, sep=sep)
        except Exception:
            continue
        if any(c not in df.columns for c in FEATURES):
            continue
        idcol = next((c for c in PATIENT_ID_COLS if c in df.columns), None)
        if idcol is not None:
            for pid, g in df.groupby(idcol, sort=False):
                yield str(pid), g.reset_index(drop=True)
        else:
            yield Path(fp).stem, df


X_parts, y_parts, pid_parts, sep_parts = [], [], [], []
used = skipped = 0
t0 = time.time()

for pid, df in load_patient_frames(files):
    Xp, yp, idx = patient_windows(df, SEQ_LEN, HORIZON)
    if Xp is None:
        skipped += 1
        if N_PATIENTS_LIMIT and (used + skipped) >= N_PATIENTS_LIMIT:
            break
        continue
    used += 1
    X_parts.append(Xp); y_parts.append(yp)
    pid_parts.append(np.full(len(yp), pid, dtype=object))
    sep_parts.append(df["SepsisLabel"].to_numpy()[idx] if "SepsisLabel" in df.columns
                     else np.zeros(len(yp), np.int8))
    if used % 2000 == 0:
        print(f"  {used} patients | {time.time()-t0:.0f}s | "
              f"{sum(len(p) for p in y_parts)} windows")

X = np.concatenate(X_parts); y = np.concatenate(y_parts)
pids = np.concatenate(pid_parts); sepsis = np.concatenate(sep_parts).astype(np.int8)

print(f"\nDone in {time.time()-t0:.0f}s")
print(f"  patients used    : {used}  (skipped {skipped} — stay shorter than {SEQ_LEN}h)")
print(f"  windows          : {len(y)}")
print(f"  X shape          : {X.shape}")
for i, n in enumerate(CLASS_NAMES):
    print(f"    {n:9s}: {(y==i).sum():>8} ({(y==i).mean()*100:5.2f}%)")
print("  missingness      : " + ", ".join(
    f"{f} {np.isnan(X[:,:,i]).mean()*100:.1f}%" for i, f in enumerate(FEATURES)))

## 3 · Patient-wise split + imputation + scaling

Split by **patient**, never by window — otherwise the same patient appears in train and test and the metrics are meaningless.

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Patient-wise split, stratified by whether the patient ever had a critical window
crit_by_pid = pd.Series(y).groupby(pids).max()
pid_train, pid_tmp = train_test_split(
    crit_by_pid.index.values, test_size=0.30, random_state=SEED,
    stratify=crit_by_pid.values)
pid_val, pid_test = train_test_split(
    pid_tmp, test_size=0.50, random_state=SEED,
    stratify=crit_by_pid.loc[pid_tmp].values)

tr = np.isin(pids, pid_train); va = np.isin(pids, pid_val); te = np.isin(pids, pid_test)
X_tr, y_tr = X[tr], y[tr]
X_va, y_va = X[va], y[va]
X_te, y_te = X[te], y[te]
print(f"train {len(y_tr):>7} windows / {len(pid_train)} patients")
print(f"val   {len(y_va):>7} windows / {len(pid_val)} patients")
print(f"test  {len(y_te):>7} windows / {len(pid_test)} patients")

# ── Medians from TRAIN ONLY (no leakage) ──
FALLBACKS = dict(zip(FEATURES, [75.0, 98.0, 37.0, 120.0, 80.0]))
flat = X_tr.reshape(-1, len(FEATURES))
medians = {}
print("\nTraining-split medians:")
for i, f in enumerate(FEATURES):
    col = flat[:, i]
    medians[f] = FALLBACKS[f] if np.all(np.isnan(col)) else float(np.nanmedian(col))
    print(f"  {f:6s} {medians[f]:7.2f}  (missing {np.mean(np.isnan(col))*100:4.1f}%)")


def featurize(Xin, medians, scaler=None, fit=False):
    """Impute -> append 5 missing-indicators -> scale the 5 raw channels only."""
    Xw = Xin.astype(np.float32, copy=True)
    ind = np.isnan(Xw).astype(np.float32)          # 1.0 = missing
    for i, f in enumerate(FEATURES):
        col = Xw[:, :, i]
        col[np.isnan(col)] = medians[f]
    n, t, _ = Xw.shape
    flat = Xw.reshape(-1, len(FEATURES))
    flat = (scaler.fit_transform(flat) if fit else scaler.transform(flat))
    Xw = flat.reshape(n, t, len(FEATURES))
    return np.concatenate([Xw, ind], axis=2).astype(np.float32)   # (n, t, 10)


scaler = StandardScaler()
X_tr_f = featurize(X_tr, medians, scaler, fit=True)
X_va_f = featurize(X_va, medians, scaler)
X_te_f = featurize(X_te, medians, scaler)

# Labels are ALREADY 0=normal, 1=warning, 2=critical.
# Bind a LabelEncoder with that exact order for backend compatibility.
# Fitting on the names would sort them alphabetically (critical, normal,
# warning) and silently remap every class — a real bug caught in testing.
le = LabelEncoder()
le.classes_ = np.asarray(CLASS_NAMES)
assert list(le.inverse_transform([0, 1, 2])) == CLASS_NAMES, "class order mismatch"
y_tr_e, y_va_e, y_te_e = y_tr, y_va, y_te
y_tr_c = tf.keras.utils.to_categorical(y_tr_e, 3)
y_va_c = tf.keras.utils.to_categorical(y_va_e, 3)
y_te_c = tf.keras.utils.to_categorical(y_te_e, 3)

cw = compute_class_weight("balanced", classes=np.unique(y_tr_e), y=y_tr_e)
class_weight = dict(enumerate(cw))
print(f"\nFinal input shape: {X_tr_f.shape}")
print(f"Class weights    : {dict(zip(CLASS_NAMES, np.round(cw,3)))}")

## 4 · Build the model

**On "maximum parameters":** the model has 5 input signals and ~600k windows. Going much past ~150k parameters mostly buys you **overfitting to the rule engine's own labels**, not accuracy — because the labels are a deterministic function of 5 numbers. The cell below prints the train/val gap after training; if it exceeds ~0.05, go *smaller*, not larger. The real lever for your grade is the ablation in step 6, not parameter count.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

tf.random.set_seed(SEED); np.random.seed(SEED)

n_feat = X_tr_f.shape[2]
model = Sequential([
    layers.Input(shape=(SEQ_LEN, n_feat)),
    layers.LSTM(CFG["lstm"][0], return_sequences=True),
    layers.Dropout(CFG["dr"][0]),
    layers.LSTM(CFG["lstm"][1]),
    layers.Dropout(CFG["dr"][1]),
    layers.Dense(CFG["dense"], activation="relu"),
    layers.Dropout(CFG["dr"][2]),
    layers.Dense(3, activation="softmax"),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()
print(f"\nTotal parameters: {model.count_params():,}")

## 5 · Train
The checkpoint goes to Drive (if enabled) so a disconnect never costs you the run.

In [ ]:
if USE_DRIVE and IS_COLAB:
    OUT_DIR = Path("/content/drive/MyDrive/healsense") / MODEL_VERSION
else:
    OUT_DIR = Path("/kaggle/working/healsense" if IS_KAGGLE else "./models") / MODEL_VERSION
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT = str(OUT_DIR / "lstm_best.h5")
print(f"Artifacts → {OUT_DIR}")

callbacks = [
    ModelCheckpoint(CKPT, monitor="val_loss", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1),
]

# ── Runtime estimate before committing to a long fit ──
steps_per_epoch = int(np.ceil(len(y_tr) / BATCH))
_ms_per_step = 45 if gpus else 90          # measured: ~31ms GPU / 40-90ms CPU
_est_min = steps_per_epoch * _ms_per_step * EPOCHS / 1000 / 60
print(f"~{steps_per_epoch} steps/epoch  |  preset={PRESET}  |  "
      f"device={'GPU' if gpus else f'CPU x{N_CPU}'}")
print(f"Estimated training time: ~{_est_min:.0f} min for {EPOCHS} epochs "
      f"(early stopping usually cuts this).")
print("Safe to leave running — the checkpoint saves every epoch.")
print("")

history = model.fit(
    X_tr_f, y_tr_c,
    validation_data=(X_va_f, y_va_c),
    epochs=EPOCHS, batch_size=BATCH,
    class_weight=class_weight,
    callbacks=callbacks, verbose=1,
)
# Overfit check
va_acc = history.history["val_accuracy"][-1]
tr_acc = history.history["accuracy"][-1]
gap = tr_acc - va_acc
print(f"\ntrain acc {tr_acc:.4f} | val acc {va_acc:.4f} | gap {gap:+.4f}")
print("✅ Generalising well." if gap <= 0.05 else
      "⚠️  Gap > 0.05 — overfitting. Try PRESET='small' or raise dropout.")

## 6 · Evaluate — including the ablation that answers *"what does the AI add?"*

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

probs = model.predict(X_te_f, batch_size=512, verbose=0)
pred = probs.argmax(1)
model_class = np.array(CLASS_NAMES)[pred]

print("=" * 62)
print("LSTM vs FUTURE-WINDOW GROUND TRUTH")
print("=" * 62)
print(classification_report(y_te, pred, target_names=CLASS_NAMES, digits=3, zero_division=0))
print(confusion_matrix(y_te, pred))

# ── Rules-only baseline on the SAME windows ──
# "Current state" severity = what the deterministic engine would say right now.
def current_severity(Xraw):
    out = np.zeros(len(Xraw), np.int8)
    hr, spo2, temp, sbp, dbp = (Xraw[:, -1, i] for i in range(5))
    with np.errstate(invalid="ignore"):
        crit = ((spo2 < 92) | (hr > 150) | (hr < 40) | (sbp >= 180) | (dbp >= 120))
        warn = ((spo2 < 95) | (hr > 120) | (hr < 50) | (temp >= 38) | (temp < 35))
    out[np.nan_to_num(warn, nan=False).astype(bool)] = WARNING
    out[np.nan_to_num(crit, nan=False).astype(bool)] = CRITICAL
    return out

rules_now = current_severity(X_te)

print("\n" + "=" * 62)
print("ABLATION — predicted vs FUTURE labels (higher F1 = better early warning)")
print("=" * 62)
rows = []
for name, p in [("Rules only (now)", rules_now), ("LSTM only", pred)]:
    rows.append((name,
                 f1_score(y_te, p, average="macro", zero_division=0),
                 f1_score(y_te, p, average=None, zero_division=0)))

fusion = rules_now.copy()
esc = (pred == CRITICAL) & (rules_now != CRITICAL)
fusion[esc] = np.where(rules_now[esc] == NORMAL, WARNING, rules_now[esc])   # capped escalation
rows.append(("Fusion (rules + LSTM, capped)",
             f1_score(y_te, fusion, average="macro", zero_division=0),
             f1_score(y_te, fusion, average=None, zero_division=0)))

print(f"{'Method':34s} {'macro-F1':>9s}  " + "  ".join(f"{n[:8]:>9s}" for n in CLASS_NAMES))
for name, mf1, per in rows:
    print(f"{name:34s} {mf1:9.3f}  " + "  ".join(f"{v:9.3f}" for v in per))

# ── The headline metric: early warning capture ──
pre_det = (y_te == CRITICAL) & (rules_now == NORMAL)
if pre_det.sum() > 0:
    caughtted = ((model_class == "critical") | (model_class == "warning"))[pre_det].mean()
    print(f"\nEARLY-WARNING CAPTURE (the number for your report)")
    print(f"  windows heading toward a critical state within {HORIZON}h, where the")
    print(f"  rules engine currently reports NORMAL : {pre_det.sum():,} windows")
    print(f"  fraction the LSTM already flags as warning/critical: {caughtted*100:.1f}%")
    print(f"  fraction the rules engine flags                     : 0.0%  (by definition)")

## 7 · Save artifacts — drop-in compatible with the backend

In [ ]:
import joblib, shutil

# The dataset calls them HR/O2Sat/Temp/SBP/DBP; the backend calls them
# heart_rate/spo2/temperature/systolic_bp/diastolic_bp. Persist the CANONICAL
# names so the artifact drops straight into PredictionService without a
# translation layer (column order is unchanged, only the keys).
CANONICAL = ["heart_rate", "spo2", "temperature", "systolic_bp", "diastolic_bp"]
CANON = dict(zip(FEATURES, CANONICAL))
medians_canon = {CANON[k]: v for k, v in medians.items()}
fallbacks_canon = {CANON[k]: v for k, v in FALLBACKS.items()}

joblib.dump(scaler, OUT_DIR / "scaler.pkl")
joblib.dump(le, OUT_DIR / "label_encoder.pkl")
joblib.dump({
    "feature_order": CANONICAL,          # backend column names
    "dataset_columns": FEATURES,         # original PhysioNet names, for traceability
    "medians": medians_canon,
    "clinical_fallbacks": fallbacks_canon,
    "strategy": "training_median",
    "include_missing_indicators": True,
    "sequence_length": SEQ_LEN,
    "n_features": len(FEATURES) * 2,
    "model_version": MODEL_VERSION,
}, OUT_DIR / "imputer.pkl")

meta = {
    "model_version": MODEL_VERSION, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "source": "PhysioNet/CinC Challenge 2019 v1.0.0 (CC BY 4.0)",
    "preset": PRESET, "params": int(model.count_params()),
    "sequence_length": SEQ_LEN, "horizon_hours": HORIZON,
    "feature_order": FEATURES, "class_names": CLASS_NAMES,
    "include_missing_indicators": True, "n_input_features": n_feat,
    "label_scheme": "future_window_max_severity",
    "imputation": {"strategy": "training_median", "medians": medians},
    "data": {"train_windows": int(len(y_tr)), "val_windows": int(len(y_va)),
             "test_windows": int(len(y_te)),
             "train_patients": int(len(pid_train)), "val_patients": int(len(pid_val)),
             "test_patients": int(len(pid_test))},
    "metrics": {"val_accuracy": float(va_acc), "train_val_gap": float(gap),
                "test_macro_f1": float(f1_score(y_te, pred, average="macro", zero_division=0))},
    "known_limitations": [
        "Labels are rules-derived future-window proxies, not clinical outcome labels.",
        "Missing indicators are fed to the model; medians compress variance of missing features.",
    ],
}
with open(OUT_DIR / "model_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved to", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:22s} {p.stat().st_size/1e3:8.1f} KB")

# Zip for download
if not USE_DRIVE or IS_KAGGLE:
    shutil.make_archive("/content/healsense_model" if IS_COLAB else "./healsense_model",
                        "zip", str(OUT_DIR))
    print("\nZipped → download it and commit to data/models/" + MODEL_VERSION)

## 8 · Smoke-test the artifacts (optional)

Verifies the exported files load and inference works with **missing BP**, the exact failure mode that broke the old pipeline.

In [ ]:
import joblib, numpy as np, tensorflow as tf

m = tf.keras.models.load_model(OUT_DIR / "lstm_best.h5")
imp = joblib.load(OUT_DIR / "imputer.pkl")
sc = joblib.load(OUT_DIR / "scaler.pkl")
enc = joblib.load(OUT_DIR / "label_encoder.pkl")

def predict_history(history):
    """history: list of dicts. Missing keys imputed with training medians."""
    raw, ind = [], []
    for step in history:
        r, i = [], []
        for f in imp["feature_order"]:
            v = step.get(f)
            miss = v is None or (isinstance(v, float) and np.isnan(v))
            i.append(1.0 if miss else 0.0)
            r.append(imp["medians"][f] if miss else float(v))
        raw.append(r); ind.append(i)
    raw = np.array([raw], np.float32); ind = np.array([ind], np.float32)
    n, t, _ = raw.shape
    raw = sc.transform(raw.reshape(-1, 5)).reshape(n, t, 5)
    return enc.inverse_transform(m.predict(np.concatenate([raw, ind], 2), verbose=0).argmax(1))[0]

# 24 readings, BP never measured — should still predict
# NOTE: canonical backend keys (heart_rate/spo2/...), not the raw dataset names.
demo = [{"heart_rate": 88, "spo2": 96, "temperature": 37.2}
        for _ in range(SEQ_LEN)]
print("Missing BP    →", predict_history(demo))
demo = [{"heart_rate": 140, "spo2": 89, "temperature": 38.9}
        for _ in range(SEQ_LEN)]
print("Deteriorating →", predict_history(demo))
print("\n✅ Artifacts are backend-compatible.")

---
### Reproducibility notes for your report

- **Dataset:** PhysioNet/CinC Challenge 2019 v1.0.0, 40,336 ICU stays, CC BY 4.0, open access.
- **Split:** patient-wise 70/15/15, stratified on whether the patient ever reaches a critical window — no window from a test patient appears in training.
- **Label:** `max(severity(t+1 .. t+K))` with `K = 6`, severity from the same WHO thresholds the backend rules engine enforces.
- **Imputation:** per-feature training-split medians, plus 5 binary missing-indicator channels so the model can distinguish "37.0 °C measured" from "37.0 °C imputed".
- **Known limitation:** labels remain rules-derived proxies. Report the ablation, not a diagnostic claim.